# Preprocesamiento

Apartado del enunciado: **3.1**, mínimo tres técnicas de preprocesamiento, indicando qué problemática
resuelve cada una.

Este cuaderno define y justifica los preprocesados que se compararán después con todos los modelos.

## Planteamiento

Se definen una **referencia** y **preprocesados incrementales**. Cada uno añade **una sola técnica** al
anterior. Así, la diferencia de métricas entre dos preprocesados consecutivos se puede atribuir a la
técnica añadida, y no a una mezcla de cambios.

| ID | Técnica que añade | Problemática del EDA que resuelve |
| :--- | :--- | :--- |
| `P0` | Referencia: dummies y mediana (no cuenta como técnica) | Ninguna. Es la base de comparación |
| `P1` | **Técnica 1: limpieza clínica** + StandardScaler | Valores imposibles, menores, `colesterol_total` incoherente, ejercicio no entero, nulos, escalas distintas |
| `P2` | Técnica 2: **por decidir** | Ver "Problemáticas pendientes" al final |
| `P3` | Técnica 3: **por decidir** | Ver "Problemáticas pendientes" al final |



## 1. Librerías y configuración

Se fija la **semilla** para que los resultados sean reproducibles y se reúnen los **umbrales de dominio**
en una sola celda. Cada umbral sale del EDA (`01_eda.ipynb`) o de la literatura citada en la sección 6.2.

In [10]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", None, "display.width", 200)

SEED = 123
DATA_PATH = "../data/raw/enfermedad_cardiovascular.csv"   # se lee sin modificarlo
TARGET = "riesgo_cardiovascular"

# --- Umbrales del EDA ---
EDAD_MINIMA = 18          # el enunciado limita la población a adultos
IMC_MINIMO = 13           # por debajo de 12-13 es biológicamente imposible (mínimo en los datos: 3.6)
HBA1C_MINIMA = 3          # HbA1c < 3 % no es fisiológica (1 caso)
EJERCICIO_MAXIMO = 7      # días por semana
SISTOLICA_MINIMA = 60
DIASTOLICA_MINIMA = 40

NOMINALES = ["tipo_alimentacion", "antecedentes_familiares_tipo"]
ORDINAL = "nivel_actividad"
ORDEN_ACTIVIDAD = {"Sedentario": 0, "Ligeramente activo": 1, "Moderadamente activo": 2, "Muy activo": 3}

## 2. Carga de datos y partición hold-out

- **Partición estratificada 80/20** con la semilla fija. Mantiene la proporción de clases en train y
  test, y el test es el mismo para todos los preprocesados y modelos, así que sus métricas son
  comparables.
- El test **solo debe contener adultos**, que es la población del enunciado. Se comprueba que no cae
  ningún menor en él.
- El test se reserva para la evaluación final. Los preprocesados y los modelos **se eligen con
  validación cruzada sobre el train**.

In [11]:
df = pd.read_csv(DATA_PATH)
X, y = df.drop(columns=TARGET), df[TARGET]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_te, y_te = X_te[X_te["edad"] >= EDAD_MINIMA], y_te[X_te["edad"] >= EDAD_MINIMA]

print("Total", X.shape, "| train", X_tr.shape, "| test", X_te.shape)
print("Menores: en train", int((X_tr["edad"] < EDAD_MINIMA).sum()), "| en test", int((X["edad"] < EDAD_MINIMA).sum() - (X_tr["edad"] < EDAD_MINIMA).sum()))
pd.DataFrame({"train": y_tr.value_counts(normalize=True), "test": y_te.value_counts(normalize=True)}).sort_index().round(3)

Total (2254, 27) | train (1803, 27) | test (451, 27)
Menores: en train 4 | en test 0


,train,test
riesgo_cardiovascular,,
0,0.5,0.501
1,0.3,0.299
2,0.2,0.200


## 3. Reglas de dominio: qué valores son imposibles

Se traducen a código las conclusiones del EDA:

- **Celdas imposibles** (`mascaras_invalidas`): HDL y triglicéridos <= 0, alcohol < 0, IMC < 13,
  HbA1c < 3, ejercicio > 7 días, presión imposible (sección 5.2) y más años fumando que edad.
- **Filas que no deben usarse para entrenar** (`filas_a_eliminar`): menores, porque el enunciado limita la
  población a adultos, y el paciente con más años fumando que edad, porque el registro entero es
  incoherente.

Son reglas fijas que no aprenden nada de los datos, así que aplicarlas fuera del pipeline no produce fugas.

In [12]:
def mascaras_invalidas(X):
    """Devuelve {columna: máscara booleana} con las celdas imposibles según el EDA."""
    invertida = X["presion_sistolica"] <= X["presion_diastolica"]
    return {
        "colesterol_hdl": X["colesterol_hdl"] <= 0,
        "trigliceridos": X["trigliceridos"] <= 0,
        "consumo_alcohol_semanal": X["consumo_alcohol_semanal"] < 0,
        "indice_masa_corporal": X["indice_masa_corporal"] < IMC_MINIMO,
        "frecuencia_ejercicio_semanal": X["frecuencia_ejercicio_semanal"] > EJERCICIO_MAXIMO,
        "hemoglobina_glicosilada": X["hemoglobina_glicosilada"] < HBA1C_MINIMA,
        "presion_sistolica": invertida | (X["presion_sistolica"] < SISTOLICA_MINIMA),
        "presion_diastolica": invertida | (X["presion_diastolica"] < DIASTOLICA_MINIMA),
        "anos_fumador": X["anos_fumador"] > X["edad"],
    }


def filas_a_eliminar(X):
    """Menores y pacientes con más años fumando que edad."""
    return (X["edad"] < EDAD_MINIMA) | (X["anos_fumador"] > X["edad"])

**Recuento de problemas en el dataset completo.** Junto a los imposibles se muestran los nulos, que también
hay que tratar.

In [13]:
mascaras = mascaras_invalidas(X)
resumen = pd.DataFrame({
    "imposibles": {c: int(m.sum()) for c, m in mascaras.items()},
    "nulos": X.isna().sum(),
}).fillna(0).astype(int)
resumen = resumen[resumen.sum(axis=1) > 0].sort_values(["imposibles", "nulos"], ascending=False)
print("Filas con algún valor imposible:", int(pd.concat(mascaras, axis=1).any(axis=1).sum()), "de", len(X))
print("Filas que no se usan para entrenar:", int(filas_a_eliminar(X).sum()))
resumen

Filas con algún valor imposible: 279 de 2254
Filas que no se usan para entrenar: 4


,imposibles,nulos
consumo_alcohol_semanal,177,0
presion_diastolica,47,0
presion_sistolica,44,0
frecuencia_ejercicio_semanal,41,0
indice_masa_corporal,19,0
colesterol_hdl,4,0
trigliceridos,4,0
anos_fumador,1,0
hemoglobina_glicosilada,1,0
horas_sueno_diarias,0,180


## 4. Criterios generales

### 4.1 Atípicos: solo se corrige lo imposible

**Decisión:** un valor solo se modifica si es imposible según las reglas de la sección 3. El resto de
atípicos **se conserva**.

**Justificación:**
- En medicina, los valores extremos (PCR, glucosa o presión muy altas) suelen ser precisamente los pacientes
  de **alto riesgo**, que es la clase que más interesa detectar. Eliminarlos o recortarlos quitaría la
  señal más útil.
- Su posible efecto sobre los modelos sensibles a la escala (SVM, KNN, regresión logística) queda como
  problemática pendiente para los preprocesados siguientes, que tendrán que tratarlo sin borrar pacientes.

### 4.2 `nivel_actividad` se mantiene aunque sea subjetiva

**Duda:** es una variable declarada por el paciente, y la tabla siguiente muestra que la frecuencia de
ejercicio media apenas cambia entre niveles. Es decir, no cuadra del todo con el ejercicio real.

**Decisión:** se mantiene.
- El riesgo medio baja de forma **monótona** con el nivel, así que la variable es muy informativa.

En la memoria se mencionará su posible subjetividad como limitación.

In [14]:
pd.DataFrame({
    "n": X[ORDINAL].value_counts(),
    "ejercicio_medio": X.groupby(ORDINAL)["frecuencia_ejercicio_semanal"].mean(),
    "riesgo_medio": y.groupby(X[ORDINAL]).mean(),
}).loc[list(ORDEN_ACTIVIDAD)].round(2)

,n,ejercicio_medio,riesgo_medio
nivel_actividad,,,
Sedentario,687,2.66,0.98
Ligeramente activo,695,2.70,0.73
Moderadamente activo,563,3.01,0.51
Muy activo,309,2.79,0.36


## 5. P0: referencia

Lo mínimo para que cualquier modelo pueda entrenar:
- **Dummies** (`OneHotEncoder`) para las variables de texto. Incluye `nivel_actividad`, a la que aquí no se
  le impone un orden.
- **Mediana** para los nulos, porque los modelos de sklearn no admiten NaN.

Los valores imposibles se quedan tal cual. P0 no cuenta como técnica: sirve para medir cuánto aporta cada
una de las siguientes.

## 6. P1: limpieza clínica (técnica 1)

| Problemática del EDA | Tratamiento en P1 |
| :--- | :--- |
| Menores (4) y más años fumando que edad (1) | Esas filas se eliminan **solo al entrenar**; el test se predice entero |
| HDL y triglicéridos <= 0, alcohol < 0, IMC < 13, HbA1c < 3, ejercicio > 7 | Se **recortan** a su límite válido |
| Sistólica <= diastólica, sistólica < 60, diastólica < 40 | La presión pasa a **NaN** y después se imputa |
| Ejercicio con decimales | **Redondeo hacia arriba**, con límite en 7 |
| `colesterol_total` incoherente | Se **elimina** (sección 6.4) |
| `nivel_actividad` en texto | **Ordinal 0-3**, porque sus categorías tienen un orden natural |
| Variables nominales en texto | **Dummies** |
| Nulos en 4 variables | **Mediana** del train |
| Escalas muy distintas | **StandardScaler** |

La imputación, las dummies y el escalado se montarán en el `Pipeline` cuando estén decididos los
preprocesados siguientes. Aquí se definen las reglas y los transformadores propios de la limpieza.

### 6.1 ¿Por qué recortar los imposibles y no pasarlos a NaN ni eliminar la fila?

Hay tres estrategias posibles:

| Estrategia | Problema |
| :--- | :--- |
| Eliminar la fila | Se perderían 279 de 2.254 pacientes (12 %) |
| Pasar a NaN e imputar con la mediana | Borra la información que llevan esos valores |
| **Recortar al límite válido** | Conserva el orden y la señal |

Para la presión y los años fumando no existe un "límite válido" natural, porque dependen de otra variable.
Por eso pasan a NaN y se imputan.

In [15]:
# Límite válido de cada variable al recortar: (mínimo, máximo); None = sin límite por ese lado.
LIMITES_RECORTE = {
    "colesterol_hdl": (1, None),
    "trigliceridos": (1, None),
    "consumo_alcohol_semanal": (0, None),
    "indice_masa_corporal": (IMC_MINIMO, None),
    "frecuencia_ejercicio_semanal": (None, EJERCICIO_MAXIMO),
    "hemoglobina_glicosilada": (HBA1C_MINIMA, None),
}


class TratarImposibles(BaseEstimator, TransformerMixin):
    """Recorta los imposibles a su límite válido. Si la variable no tiene límite propio (presión y años
    fumando), la celda pasa a NaN. No aprende nada de los datos, por eso fit no hace nada."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for columna, mascara in mascaras_invalidas(X).items():
            if columna in LIMITES_RECORTE:
                X[columna] = X[columna].clip(*LIMITES_RECORTE[columna])
            else:
                X.loc[mascara, columna] = np.nan
        return X

### 6.2 Presión: ¿qué margen entre sistólica y diastólica es imposible?

La sistólica tiene que ser mayor que la diastólica. La duda es si, además, hace falta un **margen mínimo**
(la presión de pulso) por debajo del cual el paciente no podría estar vivo. 

**Decisión:** son imposibles la sistólica <= diastólica y los valores fuera de rango absoluto (sistólica
< 60, diastólica < 40). La tabla siguiente lo confirma con los datos: los casos con un margen positivo pero
pequeño tienen un riesgo medio parecido al resto (0,72 y 0,62 frente a 0,72), así que no se comportan como
errores. Solo los de sistólica <= diastólica se apartan (0,45).

In [16]:
pulso = X["presion_sistolica"] - X["presion_diastolica"]
tramos = pd.cut(pulso, [-np.inf, 0, 10, 25, np.inf], labels=["<= 0", "(0, 10]", "(10, 25]", "> 25"])
display(pd.DataFrame({"n": tramos.value_counts(), "riesgo_medio": y.groupby(tramos, observed=False).mean()}).round(2))
print("Sistólica < 60:", int((X["presion_sistolica"] < SISTOLICA_MINIMA).sum()),
      "| diastólica < 40:", int((X["presion_diastolica"] < DIASTOLICA_MINIMA).sum()))

,n,riesgo_medio
<= 0,44,0.45
"(0, 10]",83,0.72
"(10, 25]",295,0.62
> 25,1832,0.72


Sistólica < 60: 1 | diastólica < 40: 3


### 6.3 Frecuencia de ejercicio semanal: redondeo hacia arriba

Es un número de días, así que debería ser entero, pero el 81,5 % de los valores tiene decimales.

**Decisión:** redondear **siempre hacia arriba** (6,1 → 7, porque ha hecho ejercicio algún rato del séptimo
día), con un **límite de 7**. El límite lo aplica `TratarImposibles` y el redondeo `TransformarColumnas`.

**Efecto secundario:** la media sube alrededor de medio día.

### 6.4 Colesterol total: se elimina

**Decisión:** `colesterol_total` se **elimina**. No se recalcula con la fórmula.

**Por qué el valor registrado no sirve.** El colesterol total debería ser aproximadamente HDL + LDL + TG/5
(fórmula de Friedewald despejada). La celda siguiente muestra que:
- el valor registrado **no se relaciona** con la fórmula (correlación prácticamente nula);
- en una cuarta parte de los pacientes el total es **menor que HDL + LDL**, lo cual es imposible, porque
  HDL y LDL son partes del total;
- su correlación con el target es muy baja.

Es decir, la columna es ruido y no se puede reparar celda a celda.

**Por qué no recalcularla.** La alternativa sería sustituirla por HDL + LDL + TG/5. Se descarta:

1. **No aporta información nueva.** Sería una combinación lineal exacta de tres variables que el modelo
   ya recibe. Un modelo no puede aprender de ella nada que no pueda aprender de sus componentes.
2. **Sería un dato inventado, no medido.** Friedewald es una estimación (se usa para calcular el LDL, no
   el total) y no es válida con triglicéridos > 400 mg/dL, que aparecen en los datos.
3. **Arrastra la imputación.** LDL tiene nulos, así que habría que imputarlo antes de aplicar la fórmula:
   el total heredaría el valor imputado y añadiría un paso más que ajustar solo con el train.
4. **Tampoco mejora la señal:** la fórmula se correlaciona con el target igual de poco que el valor
   registrado.

Eliminarla deja un preprocesado más simple, sin variables redundantes y sin una configuración extra que
comparar en CV.

In [17]:
formula = X["colesterol_hdl"] + X["colesterol_ldl"] + X["trigliceridos"] / 5
validos = X["colesterol_ldl"].notna()
print("Correlación registrado-fórmula:", round(X["colesterol_total"].corr(formula), 3))
print("Total registrado < HDL + LDL:",
      f"{(X['colesterol_total'] < X['colesterol_hdl'] + X['colesterol_ldl'])[validos].mean():.1%}",
      "de los pacientes con LDL conocido")
print("Correlación con el target: registrado", round(X["colesterol_total"].corr(y), 3),
      "| fórmula", round(formula.corr(y), 3))
print("Triglicéridos > 400 (fuera del rango de Friedewald):", int((X["trigliceridos"] > 400).sum()),
      "| nulos de LDL:", int(X["colesterol_ldl"].isna().sum()))

Correlación registrado-fórmula: 0.026
Total registrado < HDL + LDL: 26.4% de los pacientes con LDL conocido
Correlación con el target: registrado 0.082 | fórmula 0.074
Triglicéridos > 400 (fuera del rango de Friedewald): 54 | nulos de LDL: 135


### 6.5 Transformador de columnas

Reúne las transformaciones de columnas de P1: redondeo del ejercicio, `nivel_actividad` ordinal y
eliminación de `colesterol_total`. No aprende nada de los datos, por eso `fit` no hace nada.

In [18]:
class TransformarColumnas(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["frecuencia_ejercicio_semanal"] = np.ceil(X["frecuencia_ejercicio_semanal"])
        X[ORDINAL] = X[ORDINAL].map(ORDEN_ACTIVIDAD).astype(float)
        return X.drop(columns="colesterol_total")

### 6.6 Efecto de la limpieza

Se aplican las dos reglas al dataset completo y se comparan mínimos, máximos y nulos de las columnas
afectadas. Los imposibles desaparecen, y solo la presión y los años fumando ganan NaN, que imputará la
mediana.

In [19]:
limpio = TransformarColumnas().fit_transform(TratarImposibles().transform(X))
cols = list(mascaras)
display(pd.DataFrame({
    "min_original": X[cols].min(), "min_limpio": limpio[cols].min(),
    "max_original": X[cols].max(), "max_limpio": limpio[cols].max(),
    "nulos_original": X[cols].isna().sum(), "nulos_limpio": limpio[cols].isna().sum(),
}).round(2))
e0, e1 = X["frecuencia_ejercicio_semanal"], limpio["frecuencia_ejercicio_semanal"]
print("Ejercicio. Antes: no enteros", f"{(e0 != e0.round()).mean():.1%}", "| media", round(e0.mean(), 2),
      "|| Después: valores", sorted(e1.unique().astype(int)), "| media", round(e1.mean(), 2))

,min_original,min_limpio,max_original,max_limpio,nulos_original,nulos_limpio
colesterol_hdl,-8.80,1.0,106.30,106.30,0,0
trigliceridos,-35.70,1.0,452.70,452.70,0,0
consumo_alcohol_semanal,-5.00,0.0,15.30,15.30,0,0
indice_masa_corporal,3.60,13.0,50.60,50.60,0,0
frecuencia_ejercicio_semanal,0.00,0.0,10.00,7.00,0,0
hemoglobina_glicosilada,2.82,3.0,8.78,8.78,0,0
presion_sistolica,52.20,65.4,231.36,231.36,0,44
presion_diastolica,30.80,43.7,122.50,117.90,0,47
anos_fumador,0.00,0.0,39.90,39.90,0,1


Ejercicio. Antes: no enteros 81.5% | media 2.78 || Después: valores [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)] | media 3.17
